# 02 - EDA Fear & Greed
Explore sentiment behavior over time and seasonality patterns.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pandas.plotting import autocorrelation_plot

from src.data_loader import load_raw_datasets
from src.preprocessing import preprocess_fear_greed

In [ ]:
project_root = Path('..').resolve()
fig_dir = project_root / 'outputs' / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

fg_raw, _ = load_raw_datasets(
    fear_greed_path=project_root / 'data' / 'raw' / 'fear_greed_index.csv',
    trades_path=project_root / 'data' / 'raw' / 'hyperliquid_trades.csv'
)
fg, _ = preprocess_fear_greed(fg_raw)
fg.head()

In [ ]:
# 1) Time series with sentiment score
plt.figure(figsize=(12, 5))
plt.plot(fg['Date'], fg['sentiment_score'], color='navy', label='Sentiment score')
plt.title('Fear & Greed Sentiment Score Over Time')
plt.xlabel('Date')
plt.ylabel('Score (1-5)')
plt.legend()
plt.annotate('Insight: sentiment exhibits persistent regimes.', xy=(0.01, 0.02), xycoords='axes fraction')
plt.tight_layout()
plt.savefig(fig_dir / 'nb2_sentiment_timeseries.png', dpi=300)
plt.show()

In [ ]:
# 2) Distribution by class
order = ['Extreme Fear','Fear','Neutral','Greed','Extreme Greed']
plt.figure(figsize=(10,5))
sns.countplot(data=fg, x='Classification', order=order)
plt.title('Distribution of Days by Sentiment')
plt.xlabel('Classification')
plt.ylabel('Days')
plt.xticks(rotation=20)
plt.annotate('Insight: class balance is uneven across market cycles.', xy=(0.01, 0.02), xycoords='axes fraction')
plt.tight_layout()
plt.savefig(fig_dir / 'nb2_sentiment_distribution.png', dpi=300)
plt.show()

In [ ]:
# 3) Rolling averages
fg2 = fg.copy()
fg2['roll7'] = fg2['sentiment_score'].rolling(7).mean()
fg2['roll30'] = fg2['sentiment_score'].rolling(30).mean()
plt.figure(figsize=(12,5))
plt.plot(fg2['Date'], fg2['sentiment_score'], alpha=0.3, label='Daily')
plt.plot(fg2['Date'], fg2['roll7'], label='7-day avg')
plt.plot(fg2['Date'], fg2['roll30'], label='30-day avg')
plt.title('Rolling Sentiment Averages')
plt.xlabel('Date')
plt.ylabel('Score')
plt.legend()
plt.annotate('Insight: rolling means smooth short-term mood noise.', xy=(0.01, 0.02), xycoords='axes fraction')
plt.tight_layout()
plt.savefig(fig_dir / 'nb2_sentiment_rolling.png', dpi=300)
plt.show()

In [ ]:
# 4) Autocorrelation
plt.figure(figsize=(10,4))
autocorrelation_plot(fg['sentiment_score'])
plt.title('Autocorrelation of Sentiment Score')
plt.xlabel('Lag')
plt.ylabel('Autocorrelation')
plt.annotate('Insight: sentiment has measurable persistence.', xy=(0.01, 0.02), xycoords='axes fraction')
plt.tight_layout()
plt.savefig(fig_dir / 'nb2_sentiment_autocorr.png', dpi=300)
plt.show()

In [ ]:
# 5) Monthly heatmap (year x month)
heat = fg.copy()
heat['year'] = heat['Date'].dt.year
heat['month_num'] = heat['Date'].dt.month
pivot = heat.pivot_table(index='year', columns='month_num', values='sentiment_score', aggfunc='mean')
plt.figure(figsize=(12,5))
sns.heatmap(pivot, cmap='YlGnBu', annot=False)
plt.title('Monthly Average Sentiment Heatmap')
plt.xlabel('Month')
plt.ylabel('Year')
plt.annotate('Insight: macro mood phases emerge seasonally.', xy=(0.01, 0.02), xycoords='axes fraction')
plt.tight_layout()
plt.savefig(fig_dir / 'nb2_sentiment_monthly_heatmap.png', dpi=300)
plt.show()